# Monitor simulations across options

## Print simulation directories

In [1]:
import shutil
import pandas as pd
import numpy as np
from lucifex.io import create_dir_path, find_dir_paths, dir_path_is_running
from lucifex.utils.py_utils import FrozenDict

NUMERICAL_PARAMS = FrozenDict(
    c_stabilization=None,
    c_limits=True,
)

DIR_ROOT = create_dir_path(
    NUMERICAL_PARAMS, 
    dir_root='./',
    dir_prefix='data', 
    dir_params=NUMERICAL_PARAMS.keys(), 
)

N_STOP = 50_000
INCLUDE = f'n_stop={N_STOP}_*'

sim_dir_paths_all = find_dir_paths(DIR_ROOT, include=INCLUDE)
sim_dir_paths_finished = find_dir_paths(
    DIR_ROOT, 
    include=INCLUDE,
    contains=('CHECKPOINT.h5', 'c.npz')
)
dt_thresh = 60 * 30
sim_dir_paths_running = [
    i for i in sim_dir_paths_all if dir_path_is_running(i, dt_thresh) and not i in sim_dir_paths_finished
]
sim_dir_paths_stalled = [
    i for i in sim_dir_paths_all if not i in (*sim_dir_paths_finished, *sim_dir_paths_running)
]

print_paths = lambda paths: print(*[i for i in paths], sep='\n') if paths else print(None)

print('Finished simulations:')
print_paths(sim_dir_paths_finished)

print('Running simulations:')
print_paths(sim_dir_paths_running)

print('Stalled simulations:')
print_paths(sim_dir_paths_stalled)

Finished simulations:
./data__c_stabilization=None|c_limits=True/n_stop=50000__Ra=100.0|Da=100.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__5e822ff46ee05fa4/
./data__c_stabilization=None|c_limits=True/n_stop=50000__Ra=500.0|Da=1.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__795c9d7125905bce/
./data__c_stabilization=None|c_limits=True/n_stop=50000__Ra=500.0|Da=10.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__c4deca4f8fe27111/
./data__c_stabilization=None|c_limits=True/n_stop=50000__Ra=500.0|Da=100.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__cdefe0d77fe0ec33/
./data__c_stabilization=None|c_limits=True/n_stop=50000__Ra=500.0|Da=1000.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=160|Ny=160__e5615d686d9c78ab/
./data__c_stabilization=None|c_limits=True/n_stop=50000__Ra=1000.0|Da=1.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=200|Ny=200__9563b4a91df22147/
./data__c_stabilization=None|c_limits=True/n_

## Print simulation status tables

In [2]:
extract_param = lambda dp, p, tp=str: tp(dp.split(f"{p}=")[1].split("|")[0])
status_icons = {'finished': "✅", "running": "🔄", "stalled": "⚠️", "dne": "❌"}
table_data: dict[float, list[tuple[float, float, str]]] = {}

for dir_path in sim_dir_paths_all:
    sr, Ra, Da = (extract_param(dir_path, p) for p in ('sr', 'Ra', 'Da'))
    if dir_path in sim_dir_paths_finished:
        status = status_icons['finished']
    if dir_path in sim_dir_paths_running:
        status = status_icons['running']
    if dir_path in sim_dir_paths_stalled:
        status = status_icons['stalled']
    tup = (Ra, Da, status)
    if sr not in table_data:
        table_data[sr] = [tup]
    else:
        table_data[sr].append(tup)

for sr, Ra_Da_status in table_data.items():
    Ra_axis = np.sort(np.unique([Ra for Ra, *_ in Ra_Da_status]))
    Da_axis = np.sort(np.unique([Da for _, Da, _ in Ra_Da_status]))
    Ra_Da_ticks = np.full((len(Ra_axis), len(Da_axis)), status_icons['dne'])
    for i in range(Ra_Da_ticks.shape[0]):
        for j in range(Ra_Da_ticks.shape[1]):
            for icon in status_icons.values():
                if (Ra_axis[i], Da_axis[j], icon) in Ra_Da_status:
                    Ra_Da_ticks[i, j] = icon

    Ra_Da_df = pd.DataFrame(
        Ra_Da_ticks,
        index=pd.MultiIndex.from_product([["Ra"], Ra_axis]),
        columns=pd.MultiIndex.from_product([["Da"], Da_axis]),
    )
    Ra_Da_df.columns.names = [f'sr = {sr}', '']
    print(Ra_Da_df, '\n')

sr = 0.2   Da                  
          1.0 10.0 100.0 1000.0
Ra 100.0    ❌    ❌     ✅      ❌
   1000.0   ✅    ✅     ✅      ✅
   2000.0   ✅    ✅     ✅      ✅
   500.0    ✅    ✅     ✅      ✅ 

sr = 0.1     Da
          100.0
Ra 1000.0     ✅ 

sr = 0.05    Da
          100.0
Ra 1000.0     ✅ 



## Delete stalled simulation directories

In [ ]:
CONFIRM = False

for dir_path in sim_dir_paths_stalled:
    print(f'Selected for deletion {dir_path}')
    if CONFIRM:
        try:
            shutil.rmtree(dir_path)
            print(f"Deleted {dir_path}")
        except FileNotFoundError:
            print(f"Does not exist {dir_path}")
        except PermissionError:
            print("Permission denied")
    else:
        if dir_path == sim_dir_paths_stalled[-1]:
            print('Set `CONFIRM = True` to delete')

Selected for deletion ./data/n_stop=50000__Ra=2000.0|Da=10.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=200|Ny=200__9e9866c2e47fb0c9/
Does not exist ./data/n_stop=50000__Ra=2000.0|Da=10.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=200|Ny=200__9e9866c2e47fb0c9/
Selected for deletion ./data/n_stop=50000__Ra=2000.0|Da=100.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=200|Ny=200__b6e58402b2d5afaf/
Does not exist ./data/n_stop=50000__Ra=2000.0|Da=100.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=200|Ny=200__b6e58402b2d5afaf/
Selected for deletion ./data/n_stop=50000__Ra=2000.0|Da=1000.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=200|Ny=200__58972afbae9e685a/
Does not exist ./data/n_stop=50000__Ra=2000.0|Da=1000.0|epsilon=0.01|zeta0=0.9|sr=0.2|cr=0.0|aspect=2.0|Nx=200|Ny=200__58972afbae9e685a/
